# HMM and Ridge Long-Short Regimes

Applies four-factor HMM regime signals to a long-short portfolio.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
HMM LONG-SHORT REGIME STRATEGY: QQQ/SH IMPLEMENTATION
======================================================

STRATEGY OVERVIEW:
-----------------
This strategy uses Hidden Markov Model (HMM) regime detection to dynamically
allocate between long and short positions:

POSITION ALLOCATION BY REGIME:
- BULLISH:  120% QQQ (Nasdaq-100) + 0% SH   → Net Long Exposure
- BEARISH:  0% QQQ + 12.5% SH (Short S&P)   → Net Short Exposure
- NEUTRAL:  0% QQQ + 0% SH                  → Cash Position

INSTRUMENT DETAILS:
- QQQ: Invesco QQQ Trust (tracks Nasdaq-100 index)
- SH: ProShares Short S&P 500 (inverse ETF, gains when S&P falls)
  * When we BUY SH, we gain SHORT exposure to the S&P 500
  * SH moves inversely to SPY (S&P 500)

WALK-FORWARD VALIDATION:
- Regime detection uses SPY data (broader market signal)
- HMM trained on expanding windows with no look-ahead bias
- Trading executed on QQQ (long) and SH (short proxy)

EXECUTION LOGIC:
- Signal generated at Close[t] based on regime prediction
- Execution at Open[t+1]
- Portfolio valued at Open[t+1] on execution days
- Holdings valued at Close[t] during holding periods
- Returns: Close[t]/Close[t-1] - 1 during holding periods

Author: Production Trading System
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional, List
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
from dataclasses import dataclass

warnings.filterwarnings("ignore")


class StrategyConfig:
    """Centralized configuration for all strategy parameters."""

    START_DATE = "2019-01-01"
    END_DATE = "2024-12-31"
    INITIAL_CAPITAL = 100000.0
    COMMISSION_RATE = 0.001
    QQQ_LONG_EXPOSURE = 1.2
    SH_SHORT_EXPOSURE = 0.125
    REGIME_DETECTION_TICKER = "SPY"
    LONG_TICKER = "QQQ"
    SHORT_TICKER = "SH"
    VIX_TICKER = "^VIX"
    BENCHMARK_TICKER = "URTH"
    ROLLING_VOL_WINDOW = 20
    SMA_SHORT = 20
    SMA_LONG = 200
    HMM_N_STATES = 3
    HMM_N_ITER = 100
    HMM_RANDOM_STATE = 42
    RIDGE_ALPHA = 1.0
    RIDGE_RANDOM_STATE = 42
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63


class DataFetcher:
    """Handles all data acquisition from Yahoo Finance."""

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical OHLCV data from Yahoo Finance.

        IMPORTANT: No forward-filling or back-filling of missing data.
        We use ONLY the data that exists to maintain data integrity.

        Parameters:
        -----------
        ticker : str
            Ticker symbol to fetch
        start_date : str
            Start date in YYYY-MM-DD format
        end_date : str
            End date in YYYY-MM-DD format

        Returns:
        --------
        pd.DataFrame with columns: Date, Open, High, Low, Close, Volume
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data:
                raise ValueError(f"No data returned for {ticker}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            print(
                f"✓ Fetched {len(df)} records for {ticker} ({df['Date'].min().date()} to {df['Date'].max().date()})"
            )
            return df
        except Exception as e:
            print(f"✗ Error fetching data for {ticker}: {str(e)}")
            raise


class FeatureEngine:
    """
    Calculates technical features for regime detection.

    CRITICAL: All calculations avoid look-ahead bias by using only past data.
    Features are lagged by 1 day before use in predictions.
    """

    @staticmethod
    def calculate_rolling_volatility(returns: pd.Series, window: int) -> pd.Series:
        """
        Calculate rolling volatility (annualized).

        Uses standard deviation of returns over rolling window.
        Annualized by multiplying by sqrt(252) trading days.
        """
        return returns.rolling(window=window, min_periods=window).std() * np.sqrt(252)

    @staticmethod
    def calculate_garman_klass_volatility(df: pd.DataFrame, window: int) -> pd.Series:
        """
        Calculate Garman-Klass volatility estimator.

        More efficient than close-to-close volatility as it uses
        intraday high/low information.

        Formula: σ² = 0.5 * (ln(H/L))² - (2*ln(2) - 1) * (ln(C/O))²
        """
        log_hl = np.log(df["High"] / df["Low"])
        log_co = np.log(df["Close"] / df["Open"])
        variance_daily = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2
        volatility_daily = np.sqrt(variance_daily)
        gk_vol = volatility_daily.rolling(window=window, min_periods=window).mean() * np.sqrt(252)
        return gk_vol

    @staticmethod
    def calculate_sma_crossover(close: pd.Series, short_window: int, long_window: int) -> pd.Series:
        """
        Calculate normalized SMA crossover signal.

        Returns continuous measure of short vs long term trend:
        - Positive: Short-term SMA above long-term (uptrend)
        - Negative: Short-term SMA below long-term (downtrend)
        - Magnitude indicates strength of trend
        """
        sma_short = close.rolling(window=short_window, min_periods=short_window).mean()
        sma_long = close.rolling(window=long_window, min_periods=long_window).mean()
        return (sma_short - sma_long) / sma_long

    @staticmethod
    def prepare_features(
        df_main: pd.DataFrame, df_vix: pd.DataFrame, config: StrategyConfig
    ) -> pd.DataFrame:
        """
        Prepare all features with proper lagging to avoid look-ahead bias.

        Process:
        1. Calculate raw features from OHLCV data
        2. Merge VIX data (volatility regime indicator)
        3. LAG all features by 1 day (T-1 to predict T)

        This ensures that predictions for day T use only information
        available at close of day T-1.
        """
        df = df_main.copy()
        df["Returns_CC"] = df["Close"].pct_change()
        df["Returns_OO"] = df["Open"].pct_change()
        df["RollingVol"] = FeatureEngine.calculate_rolling_volatility(
            df["Returns_CC"], config.ROLLING_VOL_WINDOW
        )
        df["GarmanKlass"] = FeatureEngine.calculate_garman_klass_volatility(
            df, config.ROLLING_VOL_WINDOW
        )
        df["SMACross"] = FeatureEngine.calculate_sma_crossover(
            df["Close"], config.SMA_SHORT, config.SMA_LONG
        )
        df_vix_clean = df_vix[["Date", "Close"]].rename(columns={"Close": "VIX"})
        df = df.merge(df_vix_clean, on="Date", how="left")
        feature_cols = ["RollingVol", "GarmanKlass", "SMACross", "VIX"]
        for col in feature_cols:
            df[f"{col}_lag"] = df[col].shift(1)
        return df


@dataclass
class RegimeMapping:
    """Stores regime mapping from training period."""

    state_to_regime: Dict[int, str]
    mean_returns: Dict[int, float]
    training_end_date: pd.Timestamp


class WalkForwardHMMRegimeDetector:
    """
    HMM Regime Detector with STRICT walk-forward validation.

    NO LOOK-AHEAD BIAS GUARANTEE:
    1. HMM trained only on expanding window of past data
    2. Regime labels assigned based on TRAINING PERIOD forward returns only
    3. Scaler, Ridge models, HMM, and regime mapping FROZEN during prediction
    4. Periodic refitting as new data becomes available

    This ensures that the regime predicted for day T uses ONLY information
    available through day T-1.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.feature_cols = ["RollingVol_lag", "GarmanKlass_lag", "SMACross_lag", "VIX_lag"]
        self.ridge_feature_cols = [f"{col}_ridge" for col in self.feature_cols]

    def walk_forward_predict(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Perform walk-forward regime prediction with NO look-ahead bias.

        Walk-Forward Process:
        ---------------------
        For each prediction window:

        TRAINING PHASE (uses data up to T-1):
        1. Fit StandardScaler on training features
        2. Fit Ridge regression for feature transformation
        3. Fit HMM on Ridge-transformed features
        4. Create regime mapping using ONLY training forward returns

        PREDICTION PHASE (predict for day T):
        5. Transform day T features with FROZEN scaler/Ridge
        6. Predict regime with FROZEN HMM
        7. Map state to regime label with FROZEN mapping

        ITERATION:
        8. Move forward one day
        9. Refit models every REFIT_FREQUENCY days (quarterly)
        10. Repeat until all days predicted

        Returns:
        --------
        DataFrame with Regime predictions (no look-ahead bias)
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD REGIME DETECTION (NO LOOK-AHEAD BIAS)")
        print("=" * 80)
        df_clean = df.dropna(subset=self.feature_cols).copy().reset_index(drop=True)
        print(f"\nTotal observations after cleaning: {len(df_clean)}")
        print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
        df_clean["Regime_State"] = np.nan
        df_clean["Regime"] = None
        for col in self.ridge_feature_cols:
            df_clean[col] = np.nan
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        print(f"\nWalk-forward configuration:")
        print(f"  Minimum training: {min_train} days (~{min_train / 252:.1f} years)")
        print(f"  Refit frequency: {refit_freq} days (~{refit_freq / 63:.1f} quarters)")
        print(f"  Total predictions needed: {len(df_clean) - min_train}\n")
        folds_executed = []
        last_refit_idx = min_train
        scaler = None
        ridge_models = None
        hmm_model = None
        regime_mapping = None
        fold_num = 0
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            if scaler is None or days_since_refit >= refit_freq:
                fold_num += 1
                train_start = 0
                train_end = i
                print(f"\n{'=' * 80}")
                print(f"FOLD {fold_num}: Refitting models at index {i}")
                print(f"{'=' * 80}")
                print(f"Training window: indices [{train_start}:{train_end}]")
                print(
                    f"Training dates: {df_clean.loc[train_start, 'Date'].date()} to {df_clean.loc[train_end - 1, 'Date'].date()}"
                )
                print(f"Training size: {train_end - train_start} days\n")
                X_train_raw = df_clean.loc[train_start : train_end - 1, self.feature_cols].values
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_raw)
                print(f"✓ StandardScaler fitted on {len(X_train_scaled)} observations")
                ridge_models = {}
                for j in range(len(self.feature_cols)):
                    other_idx = [k for k in range(len(self.feature_cols)) if k != j]
                    if len(other_idx) > 0:
                        ridge = Ridge(
                            alpha=self.config.RIDGE_ALPHA,
                            random_state=self.config.RIDGE_RANDOM_STATE,
                        )
                        ridge.fit(X_train_scaled[:, other_idx], X_train_scaled[:, j])
                        ridge_models[j] = ridge
                print(f"✓ Ridge models fitted ({len(ridge_models)} features decorrelated)")
                X_train_ridge = np.zeros_like(X_train_scaled)
                for j in range(len(self.feature_cols)):
                    other_idx = [k for k in range(len(self.feature_cols)) if k != j]
                    if j in ridge_models:
                        y_pred = ridge_models[j].predict(X_train_scaled[:, other_idx])
                        X_train_ridge[:, j] = X_train_scaled[:, j] - y_pred
                    else:
                        X_train_ridge[:, j] = X_train_scaled[:, j]
                print(f"✓ Ridge features computed for training window")
                hmm_model = hmm.GaussianHMM(
                    n_components=self.config.HMM_N_STATES,
                    covariance_type="full",
                    n_iter=self.config.HMM_N_ITER,
                    random_state=self.config.HMM_RANDOM_STATE,
                )
                hmm_model.fit(X_train_ridge)
                print(f"✓ HMM fitted (converged: {hmm_model.monitor_.converged})")
                states_train = hmm_model.predict(X_train_ridge)
                returns_forward_train = (
                    df_clean.loc[train_start : train_end - 1, "Returns_OO"].shift(-1).values
                )
                state_returns = {}
                state_counts = {}
                for state in range(self.config.HMM_N_STATES):
                    mask = states_train == state
                    mask[-1] = False
                    count = mask.sum()
                    state_counts[state] = count
                    if count > 0:
                        state_returns[state] = returns_forward_train[mask].mean()
                    else:
                        state_returns[state] = 0
                sorted_states = sorted(state_returns.items(), key=lambda x: x[1])
                state_to_regime = {
                    sorted_states[0][0]: "Bearish",
                    sorted_states[1][0]: "Neutral",
                    sorted_states[2][0]: "Bullish",
                }
                regime_mapping = RegimeMapping(
                    state_to_regime=state_to_regime,
                    mean_returns=state_returns,
                    training_end_date=df_clean.loc[train_end - 1, "Date"],
                )
                print(f"✓ Regime mapping created from training forward returns:")
                for state, regime in state_to_regime.items():
                    print(
                        f"  State {state} → {regime:8s}: {state_counts[state]:4d} obs, avg forward return: {state_returns[state]:+.6f}"
                    )
                last_refit_idx = i
                folds_executed.append(
                    {
                        "fold": fold_num,
                        "train_start": train_start,
                        "train_end": train_end,
                        "train_start_date": df_clean.loc[train_start, "Date"],
                        "train_end_date": df_clean.loc[train_end - 1, "Date"],
                        "regime_mapping": regime_mapping.state_to_regime.copy(),
                        "mean_returns": regime_mapping.mean_returns.copy(),
                    }
                )
            X_current_raw = df_clean.loc[i:i, self.feature_cols].values
            X_current_scaled = scaler.transform(X_current_raw)
            X_current_ridge = np.zeros_like(X_current_scaled)
            for j in range(len(self.feature_cols)):
                other_idx = [k for k in range(len(self.feature_cols)) if k != j]
                if j in ridge_models:
                    y_pred = ridge_models[j].predict(X_current_scaled[:, other_idx])
                    X_current_ridge[:, j] = X_current_scaled[:, j] - y_pred
                else:
                    X_current_ridge[:, j] = X_current_scaled[:, j]
            for j, col in enumerate(self.ridge_feature_cols):
                df_clean.loc[i, col] = X_current_ridge[0, j]
            state_pred = hmm_model.predict(X_current_ridge)[0]
            regime_pred = regime_mapping.state_to_regime[state_pred]
            df_clean.loc[i, "Regime_State"] = state_pred
            df_clean.loc[i, "Regime"] = regime_pred
        print(f"\n{'=' * 80}")
        print(f"WALK-FORWARD COMPLETE")
        print(f"{'=' * 80}")
        print(f"Total folds executed: {len(folds_executed)}")
        print(f"Predictions generated: {df_clean['Regime'].notna().sum()}")
        regime_counts = df_clean["Regime"].value_counts()
        print(f"\nFinal regime distribution:")
        for regime in ["Bearish", "Neutral", "Bullish"]:
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime:8s}: {count:4d} days ({pct:5.1f}%)")
        return (df_clean, folds_executed)


class LongShortBacktestEngine:
    """
    Backtesting engine for LONG-SHORT strategy with two instruments.

    INSTRUMENT ALLOCATION BY REGIME:
    --------------------------------
    BULLISH:  120% QQQ (long Nasdaq-100) + 0% SH   → Net +120% exposure
    BEARISH:  0% QQQ + 12.5% SH (short S&P via SH) → Net -12.5% exposure
    NEUTRAL:  0% QQQ + 0% SH                       → 0% exposure (cash)

    EXECUTION MECHANICS:
    -------------------
    - Signal determined at Close[t] based on regime
    - Execution at Open[t+1] (realistic 1-day lag)
    - Portfolio valued at Close[t] during holding periods
    - Commissions applied to each trade leg (QQQ and SH separately)

    IMPORTANT:
    - 120% QQQ allocation implies leverage (margin used)
    - SH is an inverse ETF: when S&P falls, SH rises (provides short exposure)
    - All positions closed before regime transitions
    """

    def __init__(
        self,
        initial_capital: float,
        commission_rate: float,
        long_exposure: float,
        short_exposure: float,
    ):
        """
        Parameters:
        -----------
        initial_capital : float
            Starting capital
        commission_rate : float
            Commission rate applied to each trade (e.g., 0.001 = 0.1%)
        long_exposure : float
            Allocation to long instrument in Bullish regime (e.g., 1.20 = 120%)
        short_exposure : float
            Allocation to short instrument in Bearish regime (e.g., 0.125 = 12.5%)
        """
        self.initial_capital = initial_capital
        self.commission_rate = commission_rate
        self.long_exposure = long_exposure
        self.short_exposure = short_exposure
        self.trades = []
        print(f"\nLong-Short Backtest Engine initialized:")
        print(f"  Initial capital: ${initial_capital:,.2f}")
        print(f"  Commission rate: {commission_rate:.3%}")
        print(f"  Bullish regime → Long exposure:  {long_exposure:>6.1%}")
        print(f"  Bearish regime → Short exposure: {short_exposure:>6.1%}")

    def run_backtest(
        self, df_regimes: pd.DataFrame, df_long: pd.DataFrame, df_short: pd.DataFrame
    ) -> pd.DataFrame:
        """
        Execute long-short backtest with proper timing and realistic execution.

        Parameters:
        -----------
        df_regimes : pd.DataFrame
            Regime predictions (from SPY analysis)
        df_long : pd.DataFrame
            Price data for long instrument (QQQ)
        df_short : pd.DataFrame
            Price data for short instrument (SH)

        Returns:
        --------
        pd.DataFrame with complete backtest results
        """
        print("\n" + "-" * 80)
        print("EXECUTING LONG-SHORT BACKTEST")
        print("-" * 80)
        results = df_regimes[["Date", "Regime", "Regime_State"]].copy()
        df_long_prices = df_long[["Date", "Open", "Close"]].rename(
            columns={"Open": "QQQ_Open", "Close": "QQQ_Close"}
        )
        results = results.merge(df_long_prices, on="Date", how="inner")
        df_short_prices = df_short[["Date", "Open", "Close"]].rename(
            columns={"Open": "SH_Open", "Close": "SH_Close"}
        )
        results = results.merge(df_short_prices, on="Date", how="inner")
        results = results.reset_index(drop=True)
        print(f"✓ Data merged: {len(results)} trading days")
        print(f"  Date range: {results['Date'].min().date()} to {results['Date'].max().date()}")
        results["Signal_QQQ"] = 0.0
        results["Signal_SH"] = 0.0
        results["Position_QQQ"] = 0.0
        results["Position_SH"] = 0.0
        results["Shares_QQQ"] = 0.0
        results["Shares_SH"] = 0.0
        results["Cash"] = 0.0
        results["Holdings_QQQ"] = 0.0
        results["Holdings_SH"] = 0.0
        results["Portfolio_Value"] = 0.0
        results["Strategy_Returns"] = 0.0
        results["Trades"] = ""
        mask_bullish = results["Regime"] == "Bullish"
        results.loc[mask_bullish, "Signal_QQQ"] = self.long_exposure
        results.loc[mask_bullish, "Signal_SH"] = 0.0
        mask_bearish = results["Regime"] == "Bearish"
        results.loc[mask_bearish, "Signal_QQQ"] = 0.0
        results.loc[mask_bearish, "Signal_SH"] = self.short_exposure
        print(f"\n✓ Signals generated:")
        print(f"  Bullish days: {mask_bullish.sum()} → {self.long_exposure:.1%} QQQ")
        print(f"  Bearish days: {mask_bearish.sum()} → {self.short_exposure:.1%} SH")
        print(f"  Neutral days: {(~mask_bullish & ~mask_bearish).sum()} → Cash")
        cash = self.initial_capital
        shares_qqq = 0.0
        shares_sh = 0.0
        results.loc[0, "Cash"] = cash
        results.loc[0, "Portfolio_Value"] = cash
        print(f"\n✓ Running daily portfolio updates...")
        for i in range(1, len(results)):
            desired_alloc_qqq = results.loc[i - 1, "Signal_QQQ"]
            desired_alloc_sh = results.loc[i - 1, "Signal_SH"]
            current_alloc_qqq = results.loc[i - 1, "Position_QQQ"]
            current_alloc_sh = results.loc[i - 1, "Position_SH"]
            trades_today = []
            if abs(desired_alloc_qqq - current_alloc_qqq) > 0.001:
                exec_price_qqq = results.loc[i, "QQQ_Open"]
                if desired_alloc_qqq > current_alloc_qqq:
                    target_value = desired_alloc_qqq * self.initial_capital
                    current_value_qqq = shares_qqq * exec_price_qqq
                    delta_value = target_value - current_value_qqq
                    shares_to_buy = delta_value / (exec_price_qqq * (1 + self.commission_rate))
                    commission = shares_to_buy * exec_price_qqq * self.commission_rate
                    cost = shares_to_buy * exec_price_qqq + commission
                    cash -= cost
                    shares_qqq += shares_to_buy
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Instrument": "QQQ",
                            "Type": "BUY",
                            "Price": exec_price_qqq,
                            "Shares": shares_to_buy,
                            "Commission": commission,
                            "Regime": results.loc[i - 1, "Regime"],
                        }
                    )
                    trades_today.append("BUY_QQQ")
                elif desired_alloc_qqq < current_alloc_qqq:
                    target_value = desired_alloc_qqq * self.initial_capital
                    current_value_qqq = shares_qqq * exec_price_qqq
                    delta_value = current_value_qqq - target_value
                    shares_to_sell = delta_value / exec_price_qqq
                    commission = shares_to_sell * exec_price_qqq * self.commission_rate
                    proceeds = shares_to_sell * exec_price_qqq - commission
                    cash += proceeds
                    shares_qqq -= shares_to_sell
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Instrument": "QQQ",
                            "Type": "SELL",
                            "Price": exec_price_qqq,
                            "Shares": shares_to_sell,
                            "Commission": commission,
                            "Regime": results.loc[i - 1, "Regime"],
                        }
                    )
                    trades_today.append("SELL_QQQ")
            if abs(desired_alloc_sh - current_alloc_sh) > 0.001:
                exec_price_sh = results.loc[i, "SH_Open"]
                if desired_alloc_sh > current_alloc_sh:
                    target_value = desired_alloc_sh * self.initial_capital
                    current_value_sh = shares_sh * exec_price_sh
                    delta_value = target_value - current_value_sh
                    shares_to_buy = delta_value / (exec_price_sh * (1 + self.commission_rate))
                    commission = shares_to_buy * exec_price_sh * self.commission_rate
                    cost = shares_to_buy * exec_price_sh + commission
                    cash -= cost
                    shares_sh += shares_to_buy
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Instrument": "SH",
                            "Type": "BUY",
                            "Price": exec_price_sh,
                            "Shares": shares_to_buy,
                            "Commission": commission,
                            "Regime": results.loc[i - 1, "Regime"],
                        }
                    )
                    trades_today.append("BUY_SH")
                elif desired_alloc_sh < current_alloc_sh:
                    target_value = desired_alloc_sh * self.initial_capital
                    current_value_sh = shares_sh * exec_price_sh
                    delta_value = current_value_sh - target_value
                    shares_to_sell = delta_value / exec_price_sh
                    commission = shares_to_sell * exec_price_sh * self.commission_rate
                    proceeds = shares_to_sell * exec_price_sh - commission
                    cash += proceeds
                    shares_sh -= shares_to_sell
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Instrument": "SH",
                            "Type": "SELL",
                            "Price": exec_price_sh,
                            "Shares": shares_to_sell,
                            "Commission": commission,
                            "Regime": results.loc[i - 1, "Regime"],
                        }
                    )
                    trades_today.append("SELL_SH")
            results.loc[i, "Shares_QQQ"] = shares_qqq
            results.loc[i, "Shares_SH"] = shares_sh
            results.loc[i, "Cash"] = cash
            results.loc[i, "Holdings_QQQ"] = shares_qqq * results.loc[i, "QQQ_Close"]
            results.loc[i, "Holdings_SH"] = shares_sh * results.loc[i, "SH_Close"]
            results.loc[i, "Portfolio_Value"] = (
                cash + results.loc[i, "Holdings_QQQ"] + results.loc[i, "Holdings_SH"]
            )
            results.loc[i, "Position_QQQ"] = results.loc[i, "Holdings_QQQ"] / self.initial_capital
            results.loc[i, "Position_SH"] = results.loc[i, "Holdings_SH"] / self.initial_capital
            results.loc[i, "Trades"] = ", ".join(trades_today) if trades_today else ""
        results["Strategy_Returns"] = results["Portfolio_Value"].pct_change()
        results["Cumulative_Returns"] = (1 + results["Strategy_Returns"]).cumprod()
        print(f"\n✓ Backtest completed:")
        print(f"  Total trades: {len(self.trades)}")
        print(f"  Final portfolio value: ${results.loc[len(results) - 1, 'Portfolio_Value']:,.2f}")
        print(
            f"  Total return: {results.loc[len(results) - 1, 'Portfolio_Value'] / self.initial_capital - 1:.2%}"
        )
        return results

    def add_buy_hold_benchmark(
        self, results: pd.DataFrame, price_df: pd.DataFrame, ticker_name: str
    ) -> pd.DataFrame:
        """
        Add buy-and-hold benchmark using Close prices.

        Parameters:
        -----------
        results : pd.DataFrame
            Main results dataframe (modified in place)
        price_df : pd.DataFrame
            Price data with Date and Close columns
        ticker_name : str
            Name for the benchmark (e.g., 'URTH', 'QQQ')
        """
        close_col = f"{ticker_name}_Close"
        if close_col not in results.columns:
            price_aligned = price_df[["Date", "Close"]].rename(columns={"Close": close_col})
            results = results.merge(price_aligned, on="Date", how="left")
        else:
            print(f"  Note: Using existing {close_col} column from backtest data")
        first_close = results.loc[0, close_col]
        shares = self.initial_capital / (first_close * (1 + self.commission_rate))
        commission_buy = shares * first_close * self.commission_rate
        col_prefix = f"{ticker_name}_BH"
        results[f"{col_prefix}_Value"] = shares * results[close_col]
        results.loc[0, f"{col_prefix}_Value"] = self.initial_capital - commission_buy
        last_idx = len(results) - 1
        last_close = results.loc[last_idx, close_col]
        commission_sell = shares * last_close * self.commission_rate
        results.loc[last_idx, f"{col_prefix}_Value"] = shares * last_close - commission_sell
        results[f"{col_prefix}_Returns"] = results[close_col].pct_change()
        results[f"{col_prefix}_Cumulative"] = (1 + results[f"{col_prefix}_Returns"]).cumprod()
        print(
            f"✓ {ticker_name} B&H benchmark added: ${results.loc[last_idx, f'{col_prefix}_Value']:,.2f}"
        )
        return results


class PerformanceMetrics:
    """Calculate comprehensive performance metrics."""

    @staticmethod
    def calculate_all_metrics(results: pd.DataFrame, rf_rate: float = 0.02) -> Dict:
        """
        Calculate performance metrics for strategy and benchmarks.

        Metrics include:
        - Total return and CAGR
        - Volatility (annualized)
        - Sharpe ratio (risk-adjusted returns)
        - Maximum drawdown
        - Win rate (% of positive return days)
        """
        metrics = {}
        strategy_returns = results["Strategy_Returns"].dropna()
        n_years = len(strategy_returns) / 252
        initial_capital = results.iloc[0]["Portfolio_Value"]
        metrics["Strategy"] = {
            "Total Return": results.iloc[-1]["Portfolio_Value"] / initial_capital - 1,
            "CAGR": (
                (results.iloc[-1]["Portfolio_Value"] / results.iloc[0]["Portfolio_Value"])
                ** (1 / n_years)
                - 1
                if n_years > 0
                else 0
            ),
            "Volatility": strategy_returns.std() * np.sqrt(252),
            "Sharpe": (
                np.sqrt(252) * (strategy_returns.mean() - rf_rate / 252) / strategy_returns.std()
                if strategy_returns.std() > 0
                else 0
            ),
            "Max Drawdown": (
                (strategy_returns + 1).cumprod()
                / (strategy_returns + 1).cumprod().expanding().max()
                - 1
            ).min(),
            "Win Rate": (strategy_returns > 0).sum() / len(strategy_returns),
            "Final Value": results.iloc[-1]["Portfolio_Value"],
        }
        for col_prefix in ["URTH_BH", "QQQ_BH"]:
            if f"{col_prefix}_Returns" in results.columns:
                bench_returns = results[f"{col_prefix}_Returns"].dropna()
                metrics[col_prefix.replace("_BH", "")] = {
                    "Total Return": results.iloc[-1][f"{col_prefix}_Value"]
                    / results.iloc[0][f"{col_prefix}_Value"]
                    - 1,
                    "CAGR": (
                        (
                            results.iloc[-1][f"{col_prefix}_Value"]
                            / results.iloc[0][f"{col_prefix}_Value"]
                        )
                        ** (1 / n_years)
                        - 1
                        if n_years > 0
                        else 0
                    ),
                    "Volatility": bench_returns.std() * np.sqrt(252),
                    "Sharpe": (
                        np.sqrt(252) * (bench_returns.mean() - rf_rate / 252) / bench_returns.std()
                        if bench_returns.std() > 0
                        else 0
                    ),
                    "Max Drawdown": (
                        (bench_returns + 1).cumprod()
                        / (bench_returns + 1).cumprod().expanding().max()
                        - 1
                    ).min(),
                    "Final Value": results.iloc[-1][f"{col_prefix}_Value"],
                }
        return metrics

    @staticmethod
    def print_metrics(metrics: Dict):
        """Print formatted metrics table."""
        print("\n" + "=" * 100)
        print("PERFORMANCE METRICS - LONG-SHORT HMM STRATEGY")
        print("=" * 100)
        for name, m in metrics.items():
            print(f"\n{name}:")
            print(f"  Total Return:     {m['Total Return']:>10.2%}")
            print(f"  CAGR:             {m['CAGR']:>10.2%}")
            print(f"  Volatility:       {m['Volatility']:>10.2%}")
            print(f"  Sharpe Ratio:     {m['Sharpe']:>10.2f}")
            print(f"  Max Drawdown:     {m['Max Drawdown']:>10.2%}")
            if "Win Rate" in m:
                print(f"  Win Rate:         {m['Win Rate']:>10.2%}")
            print(f"  Final Value:      ${m['Final Value']:>10,.2f}")
        if "Strategy" in metrics and "URTH" in metrics:
            print(f"\nRelative to URTH (Global Benchmark):")
            print(
                f"  Excess Return:    {metrics['Strategy']['Total Return'] - metrics['URTH']['Total Return']:>10.2%}"
            )
            print(
                f"  Excess CAGR:      {metrics['Strategy']['CAGR'] - metrics['URTH']['CAGR']:>10.2%}"
            )
        if "Strategy" in metrics and "QQQ" in metrics:
            print(f"\nRelative to QQQ (Long-Only Nasdaq-100):")
            print(
                f"  Excess Return:    {metrics['Strategy']['Total Return'] - metrics['QQQ']['Total Return']:>10.2%}"
            )
            print(
                f"  Excess CAGR:      {metrics['Strategy']['CAGR'] - metrics['QQQ']['CAGR']:>10.2%}"
            )
        print("=" * 100 + "\n")


def plot_results(results: pd.DataFrame, config: StrategyConfig):
    """Create comprehensive visualization plots."""
    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    regime_colors = {"Bearish": "red", "Neutral": "gray", "Bullish": "green"}
    for regime, color in regime_colors.items():
        mask = results["Regime"] == regime
        if mask.sum() > 0:
            axes[0].scatter(
                results.loc[mask, "Date"],
                results.loc[mask, "QQQ_Close"],
                c=color,
                label=regime,
                alpha=0.6,
                s=15,
            )
    axes[0].set_ylabel("QQQ Price ($)", fontsize=11)
    axes[0].set_title(
        "HMM Regime Detection (Walk-Forward Validated)", fontsize=12, fontweight="bold"
    )
    axes[0].legend(loc="upper left", fontsize=10)
    axes[0].grid(alpha=0.3)
    axes[1].fill_between(
        results["Date"],
        0,
        results["Position_QQQ"] * 100,
        alpha=0.5,
        color="green",
        label="QQQ Long",
    )
    axes[1].fill_between(
        results["Date"],
        0,
        -results["Position_SH"] * 100,
        alpha=0.5,
        color="red",
        label="SH Short (Inverse)",
    )
    axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.8)
    axes[1].set_ylabel("Exposure (% of Capital)", fontsize=11)
    axes[1].set_title("Dynamic Position Allocation by Regime", fontsize=12, fontweight="bold")
    axes[1].legend(loc="upper left", fontsize=10)
    axes[1].grid(alpha=0.3)
    axes[2].plot(
        results["Date"],
        100 * results["Cumulative_Returns"],
        label="Long-Short Strategy",
        linewidth=2.5,
        color="darkblue",
    )
    if "QQQ_BH_Cumulative" in results.columns:
        axes[2].plot(
            results["Date"],
            100 * results["QQQ_BH_Cumulative"],
            label="QQQ Buy & Hold",
            linewidth=2,
            alpha=0.7,
            color="orange",
        )
    if "URTH_BH_Cumulative" in results.columns:
        axes[2].plot(
            results["Date"],
            100 * results["URTH_BH_Cumulative"],
            label="URTH Buy & Hold",
            linewidth=2,
            alpha=0.7,
            color="purple",
        )
    axes[2].set_ylabel("Cumulative Return (Base 100)", fontsize=11)
    axes[2].set_title("Strategy vs Buy-and-Hold Benchmarks", fontsize=12, fontweight="bold")
    axes[2].legend(loc="upper left", fontsize=10)
    axes[2].grid(alpha=0.3)
    strategy_cum = (1 + results["Strategy_Returns"]).cumprod()
    strategy_dd = (strategy_cum / strategy_cum.expanding().max() - 1) * 100
    axes[3].fill_between(results["Date"], strategy_dd, 0, alpha=0.4, color="red")
    axes[3].plot(results["Date"], strategy_dd, color="darkred", linewidth=1.5)
    axes[3].set_xlabel("Date", fontsize=11)
    axes[3].set_ylabel("Drawdown (%)", fontsize=11)
    axes[3].set_title("Strategy Drawdown Profile", fontsize=12, fontweight="bold")
    axes[3].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def create_trade_analysis(results: pd.DataFrame, trades: List[Dict]) -> pd.DataFrame:
    """
    Create detailed trade analysis with regime attribution.

    Analyzes:
    - Trade frequency by regime
    - Win/loss by instrument and regime
    - Average holding periods
    """
    if len(trades) == 0:
        print("No trades to analyze")
        return pd.DataFrame()
    df_trades = pd.DataFrame(trades)
    print("\n" + "=" * 80)
    print("TRADE ANALYSIS")
    print("=" * 80)
    print("\nTrade counts by instrument:")
    print(df_trades.groupby(["Instrument", "Type"]).size())
    print("\nTrade counts by regime:")
    print(df_trades.groupby(["Regime", "Instrument", "Type"]).size())
    total_commissions = df_trades["Commission"].sum()
    print(f"\nTotal commissions paid: ${total_commissions:,.2f}")
    return df_trades


def main():
    """Main execution function for Long-Short HMM strategy."""
    print("\n" + "=" * 100)
    print("HMM LONG-SHORT REGIME STRATEGY: QQQ/SH IMPLEMENTATION")
    print("=" * 100)
    config = StrategyConfig()
    print(f"\nStrategy Configuration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.2f}")
    print(f"  Commission Rate: {config.COMMISSION_RATE:.3%}")
    print(f"\nInstruments:")
    print(f"  Regime Detection: {config.REGIME_DETECTION_TICKER} (S&P 500)")
    print(f"  Long Instrument:  {config.LONG_TICKER} (Nasdaq-100)")
    print(f"  Short Instrument: {config.SHORT_TICKER} (Inverse S&P 500)")
    print(f"  Benchmark:        {config.BENCHMARK_TICKER} (Global Equity)")
    print(f"\nExposure Levels:")
    print(f"  Bullish Regime → {config.QQQ_LONG_EXPOSURE:.1%} QQQ, {0:.1%} SH")
    print(f"  Bearish Regime → {0:.1%} QQQ, {config.SH_SHORT_EXPOSURE:.1%} SH")
    print(f"  Neutral Regime → {0:.1%} QQQ, {0:.1%} SH (Cash)")
    print(f"\nWalk-Forward Validation:")
    print(f"  Min Training: {config.MIN_TRAINING_DAYS} days")
    print(f"  Refit Frequency: {config.REFIT_FREQUENCY} days")
    print("\n" + "-" * 100)
    print("DATA ACQUISITION")
    print("-" * 100)
    fetcher = DataFetcher()
    df_spy = fetcher.fetch_historical_data(
        config.REGIME_DETECTION_TICKER, config.START_DATE, config.END_DATE
    )
    df_qqq = fetcher.fetch_historical_data(config.LONG_TICKER, config.START_DATE, config.END_DATE)
    df_sh = fetcher.fetch_historical_data(config.SHORT_TICKER, config.START_DATE, config.END_DATE)
    df_vix = fetcher.fetch_historical_data(config.VIX_TICKER, config.START_DATE, config.END_DATE)
    df_urth = fetcher.fetch_historical_data(
        config.BENCHMARK_TICKER, config.START_DATE, config.END_DATE
    )
    print("\n" + "-" * 100)
    print("FEATURE ENGINEERING")
    print("-" * 100)
    df_features = FeatureEngine.prepare_features(df_spy, df_vix, config)
    print(f"✓ Features prepared: {len(df_features)} observations")
    detector = WalkForwardHMMRegimeDetector(config)
    df_regimes, folds = detector.walk_forward_predict(df_features)
    print("\n" + "-" * 100)
    print("BACKTESTING")
    print("-" * 100)
    engine = LongShortBacktestEngine(
        initial_capital=config.INITIAL_CAPITAL,
        commission_rate=config.COMMISSION_RATE,
        long_exposure=config.QQQ_LONG_EXPOSURE,
        short_exposure=config.SH_SHORT_EXPOSURE,
    )
    results = engine.run_backtest(df_regimes, df_qqq, df_sh)
    print("\n" + "-" * 100)
    print("ADDING BENCHMARKS")
    print("-" * 100)
    results = engine.add_buy_hold_benchmark(results, df_urth, "URTH")
    results = engine.add_buy_hold_benchmark(results, df_qqq, "QQQ")
    print("\n" + "-" * 100)
    print("PERFORMANCE ANALYSIS")
    print("-" * 100)
    metrics = PerformanceMetrics.calculate_all_metrics(results)
    PerformanceMetrics.print_metrics(metrics)
    df_trades = create_trade_analysis(results, engine.trades)
    print("\n" + "=" * 80)
    print("REGIME STATISTICS")
    print("=" * 80)
    regime_stats = results.groupby("Regime").agg(
        {
            "Strategy_Returns": ["count", "mean", "std"],
            "Position_QQQ": "mean",
            "Position_SH": "mean",
        }
    )
    print("\nRegime breakdown:")
    print(regime_stats)
    print("\n" + "-" * 100)
    print("GENERATING VISUALIZATIONS")
    print("-" * 100)
    plot_results(results, config)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results.to_csv(f"longshort_results_{timestamp}.csv", index=False)
    if len(df_trades) > 0:
        df_trades.to_csv(f"longshort_trades_{timestamp}.csv", index=False)
    pd.DataFrame(folds).to_csv(f"longshort_folds_{timestamp}.csv", index=False)
    print(f"\n✓ Results exported with timestamp: {timestamp}")
    print("\n" + "=" * 100)
    print("EXECUTION COMPLETE - LONG-SHORT STRATEGY VALIDATED")
    print("=" * 100 + "\n")
    return (results, metrics, df_trades, folds)


if __name__ == "__main__":
    results, metrics, trades, folds = main()
